# 단계 5 — `structural_review` 프롬프트로 도메인 워크플로 외부화

**`structural-mcp` 누적 빌드업의 단계 ④**: 단계 4까지 만든 도구와 리소스 위에, **도메인 워크플로 자체를 캡슐화한 프롬프트**를 추가합니다.

## 본 노트북의 위치
이 노트북은 강의노트 7주차 본문 §2.7 단계 ④ 라인 1531부터 1554에 명시된 구조 검토 프롬프트 시그니처를 그대로 구현하는 단계입니다. 단계 1부터 단계 4까지 누적된 도구 세 개와 리소스 한 개 위에 프롬프트 두 개를 더 얹어, 본 단계가 끝나면 도구 세 개와 리소스 한 개와 프롬프트 두 개를 모두 갖춘 완성형 모델 컨텍스트 프로토콜 서버가 만들어집니다.

## 학습 목표
이번 단계에서는 프롬프트 데코레이터와 사용자 메시지 모듈을 사용하여 **도메인 워크플로 자체를 외부화**하는 방법을 익힙니다. 도구·리소스·프롬프트 세 가지 컴포넌트가 협업하는 패턴을 실제 코드로 시연하고, KDS 조항 인용과 SI 단위 강제와 소요/공칭 강도비 산정 같은 도메인 규약을 프롬프트가 자동으로 강제하도록 설계합니다. 한국 건축 실무에서 매우 정형화된 도메인 규약을 어떻게 컴퓨터가 직접 강제하게 만들 수 있는지를 한 번에 체험합니다.

## 선행 학습 사항
단계 4까지 만든 `structural_mcp.py` 파일에 도구 세 개와 리소스 한 개가 등록되어 있어야 합니다. 파이썬 3.11 이상과 모델 컨텍스트 프로토콜 SDK, 그리고 파이댄틱이 설치되어 있어야 합니다. 단계 6에서 통합 시연을 할 때는 앤트로픽 API 키가 필요하지만, 본 단계의 핵심 학습은 키 없이도 모두 진행할 수 있습니다.

## 본 단계 이후의 흐름
본 노트북을 마치고 나면 단계 6에서 클로드 코드 명령행 인터페이스에 등록하여 자연어로 직접 사용해 보는 흐름으로 이어집니다. 본 단계가 끝나는 시점에 만들어지는 `structural_mcp.py`는 강의노트가 정의한 모든 컴포넌트를 갖춘 완성된 서버이며, 이후 단계 6은 이 서버를 실전에서 어떻게 호출하는지 보여주는 보너스 단계입니다.


## §1. 도메인 프롬프트가 왜 필요한가

구조 검토는 자유로운 대화가 아닙니다. 다음과 같은 도메인 규약을 **매 응답마다** 일관되게 지켜야 합니다. 단위계는 반드시 국제단위계(SI)로 통일하여 뉴턴, 밀리미터, 메가파스칼을 사용해야 하고, KDS 조문 번호를 반드시 인용해야 하며, 부동소수는 둘째 자리까지 표시하고, 에러 메시지조차 한국어로 출력해야 합니다. 이러한 규약은 강의노트 §2.7 라인 1559부터 1564 사이에 명시되어 있습니다.

사용자가 매번 이러한 규칙을 직접 타이핑하는 대신, **프롬프트로 외부화**하여 서버에 저장해 둡니다. 그러면 사용자는 인자만 채워서 프롬프트를 선택하기만 하면, 항상 같은 형식의 검토 결과를 받을 수 있습니다.

> [!finding] 강의노트 §2.4의 핵심 통찰 — 프롬프트는 도메인 전문성의 외부화
> 프롬프트는 *서버 저자, 즉 이 경우 구조 전문가가 신중히 개발하고 시험한 템플릿*이므로, 사용자는 인자만 채우면 일관된 고품질 결과를 얻습니다. 이는 KDS 조문 인용과 단위 강제와 강도비 산정 의무 등 **도메인 규약이 많은 분야일수록** 더욱 강력한 효과를 발휘합니다.

세 가지 컴포넌트의 제어 주체를 비교하면 다음과 같습니다. 도구는 모델인 클로드가 능동적으로 호출하므로 일종의 쓰기 동작에 비유할 수 있고, 리소스는 애플리케이션이 필요할 때 조회하므로 읽기 동작에 비유할 수 있으며, **프롬프트는 사용자가 직접 선택하는 컴포넌트**입니다. 즉 사용자가 폼에서 항목을 고르는 것과 같은 경험을 제공합니다.

> [!tip] 한국 구조설계 실무에서의 활용
> 콘크리트구조 설계기준인 KDS 41 17 00 과 강구조 설계기준인 KDS 14 31 등 한국의 설계기준은 조문 번호 인용이 필수이고, 안전율과 강도감소계수가 매번 동일한 형식으로 적용되어야 합니다. 프롬프트로 외부화하면 *학생과 실무자와 자문 의뢰인 모두* 동일한 검토 형식을 자연어로 받을 수 있습니다.


## §2. 환경 준비 — 단계 4 산출물 다시 불러오기

단계 4까지 누적된 도구 세 개(휨 검토·전단 검토·미다스 파서)와 리소스 한 개(KDS 요약)를 그대로 재정의한 뒤, 본 단계에서는 **프롬프트만 추가**합니다. 이렇게 하면 노트북 한 개만 실행해도 그 자체로 완전한 모델 컨텍스트 프로토콜 서버가 동작하므로 학습자가 단계별로 끊어서 실행해도 무리 없이 따라올 수 있습니다.

본 셀에서는 단계 4까지의 모든 코드를 다시 정의하고, 그 위에 단계 5의 프롬프트만 새로 추가하는 방식을 따릅니다. 이러한 누적 빌드업 패턴은 학습자가 어느 단계에서 시작하더라도 항상 완전한 서버를 손에 쥘 수 있게 해 주는 본 강의의 핵심 설계 원칙입니다.

In [ ]:
# Stage 4의 structural_mcp.py를 그대로 가져와 prompt만 추가하는 누적 빌드업.
import json
import math
import os
import asyncio
from pathlib import Path
from typing import Dict, List

from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("StructuralMCP", log_level="ERROR")

# ── 도구 + 리소스 + 파서 (Stage 1~4 통합) ──
@mcp.tool(name="check_flexural_strength", description="RC 보 휨강도 검토 (KDS 41 17 00).")
def check_flexural_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    As: float = Field(description="인장 철근 단면적 (mm^2)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
    Mu: float = Field(description="소요 휨모멘트 (kN.m)"),
) -> str:
    a = As * fy / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn
    return json.dumps({"phi_Mn": round(phi_Mn, 2), "Mu": round(Mu, 2),
                       "DCR": round(Mu / phi_Mn, 3),
                       "check": "OK" if phi_Mn >= Mu else "NG"}, ensure_ascii=False)

@mcp.tool(name="check_shear_strength", description="RC 부재 전단강도 검토 (KDS 41 17 00).")
def check_shear_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    Av: float = Field(description="전단 보강근 면적 (mm^2)"),
    s: float = Field(description="전단 보강근 간격 (mm)"),
    fy: float = Field(description="전단 보강근 항복강도 (MPa)"),
    Vu: float = Field(description="소요 전단력 (kN)"),
) -> str:
    Vc = (1 / 6) * math.sqrt(fck) * b * d / 1000
    Vs = Av * fy * d / s / 1000
    phi_Vn = 0.75 * (Vc + Vs)
    return json.dumps({"phi_Vn": round(phi_Vn, 2), "Vu": round(Vu, 2),
                       "DCR": round(Vu / phi_Vn, 3),
                       "check": "OK" if phi_Vn >= Vu else "NG"}, ensure_ascii=False)

@mcp.tool(name="parse_midas_mgt", description="Parse a Midas .mgt model file.")
def parse_midas_mgt(file_path: str = Field(description="Absolute path to a .mgt file")) -> str:
    p = Path(file_path)
    if not p.is_absolute() or not p.exists():
        return json.dumps({"error": "invalid path", "given": str(p)}, ensure_ascii=False)
    sections: Dict[str, List[List[str]]] = {}
    current = None
    for raw in p.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith(";"):
            continue
        if line.startswith("*"):
            current = line[1:].split()[0].upper()
            sections.setdefault(current, [])
        elif current:
            sections[current].append([c.strip() for c in line.split(",")])
    return json.dumps({"node_count": len(sections.get("NODE", [])),
                       "element_count": len(sections.get("ELEMENT", [])),
                       "materials": sections.get("MATERIAL", [])}, ensure_ascii=False)

@mcp.resource("kds://41-17-00/summary", mime_type="application/json")
def kds_summary() -> str:
    return json.dumps({"title": "KDS 41 17 00",
                       "flexure": {"phi": 0.85, "section": "4.3.1"},
                       "shear": {"phi": 0.75, "section": "4.4"}}, ensure_ascii=False)

print("Stage 4 reloaded: 3 tools + 1 resource. Ready for prompt registration.")


## §3. 구조 검토용 프롬프트의 핵심 시그니처

강의노트 §2.7 라인 1531부터 1554에서 권장하는 **표준 시그니처**를 그대로 따릅니다. 부재 종류와 폭과 유효 깊이와 콘크리트 압축강도와 철근 항복강도, 이 다섯 개의 인자만 받습니다. 이는 한국 건축 실무에서 부재를 검토할 때 가장 먼저 확인하는 정보의 묶음과 정확히 일치합니다.

프롬프트 본문은 클로드에게 **세 가지 행동 지침**을 전달합니다. 첫째, KDS 요약 리소스를 먼저 읽어 강도감소계수와 적용 조항을 파악합니다. 둘째, 휨 검토 도구와 전단 검토 도구를 호출하여 수치 결과를 얻습니다. 필요하다면 미다스 파서 도구로 모델 파일을 미리 분석합니다. 셋째, KDS 조항 인용 형식과 SI 단위 강제와 한국어 출력을 모두 강제합니다.

> [!method] 프롬프트가 강제하는 검토 절차
> 사용자가 인자(부재 종류와 치수와 재료강도)만 채우면, 클로드는 **위 세 가지 지침을 자동으로 따릅니다**. 이는 *사용자가 직접 명령을 외울 필요 없이* 도메인 전문가의 검토 절차를 그대로 받게 됨을 의미합니다. 이 점이 모델 컨텍스트 프로토콜 프롬프트의 가장 큰 가치이며, 한국 건축 실무처럼 도메인 규약이 정형화된 분야에서 특히 빛을 발합니다.


In [ ]:
# Week_07.md §2.7 단계 ④ (라인 1531-1554) — 프롬프트 정의
@mcp.prompt(
    name="structural_review",
    description="KDS 41 17 00 기준 RC 부재 설계 검토 (도구·리소스 자동 호출).",
)
def structural_review(
    member_type: str = Field(description="부재 종류 (beam / column / slab)"),
    b: float = Field(description="부재 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
) -> list[base.Message]:
    prompt = f"""당신은 건축구조 전문가입니다. KDS 41 17 00 기준에 따라 다음 RC {member_type}을 검토하세요.

## 부재 제원
- 종류: {member_type}
- b = {b} mm, d = {d} mm
- fck = {fck} MPa, fy = {fy} MPa

## 검토 절차 (이 순서대로 도구를 호출하세요)
1. 먼저 `kds://41-17-00/summary` 리소스를 읽어 KDS 기준의 phi 계수와 적용 조항을 파악합니다.
2. `check_flexural_strength` 도구로 휨 강도를 검토합니다 (As, Mu는 사용자에게 추가 질의).
3. 필요시 `check_shear_strength` 도구로 전단 강도를 검토합니다.
4. Midas 결과 파일이 있다면 `parse_midas_mgt` 도구로 모델을 먼저 읽습니다.

## 출력 규약 (반드시 준수)
- **단위**: SI (mm, MPa, kN, kN·m). 다른 단위 사용 금지.
- **조문 인용**: 모든 결과에 \"KDS 41 17 00 4.3.1에 따라 ...\" 형식의 조항 번호를 포함.
- **수치**: 응력·강도는 소수 둘째 자리까지 표시.
- **언어**: 한국어. 에러 메시지도 한국어로.
- **첫 줄**: ✅ 만족 또는 ❌ 불만족 한 줄 판정.
"""
    return [base.UserMessage(prompt)]

print("Prompt registered: structural_review")


## §4. 추가 변형 — 부재 유형별 전용 프롬프트 (선택 사항)

부재별로 검토 항목이 다르기 때문에, 즉 보는 휨과 전단을 본격 검토하지만 기둥은 축력과 휨의 상호작용을 봐야 하고 슬래브는 양방향 휨을 다뤄야 하므로, 별도의 프롬프트로 분리할 수도 있습니다. 본 셀에서는 철근콘크리트 기둥 검토 전용으로 `column_review` 프롬프트를 예시로 추가합니다.

In [ ]:
# (선택) 부재별 검토 항목이 다르므로 column_review를 별도 프롬프트로 분리할 수도 있다.
@mcp.prompt(
    name="column_review",
    description="RC 기둥의 축력-휨 상호작용 검토 (KDS 41 17 00 4.5).",
)
def column_review(
    b: float = Field(description="기둥 단면 폭 (mm)"),
    h: float = Field(description="기둥 단면 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
    Pu: float = Field(description="소요 축력 (kN)"),
    Mu: float = Field(description="소요 휨모멘트 (kN.m)"),
) -> list[base.Message]:
    prompt = f"""당신은 구조설계 전문가입니다. KDS 41 17 00 4.5 (압축 부재) 기준으로 다음 RC 기둥을 검토하세요.

단면: {b}x{h} mm, fck={fck} MPa, fy={fy} MPa
하중: Pu={Pu} kN, Mu={Mu} kN.m

1. `kds://41-17-00/summary`로 phi 계수 확인
2. 축력-휨 상호작용도 (P-M interaction)를 단계적으로 작성
3. 첫 줄에 ✅/❌ 판정, 단위는 SI 강제, 조문 번호 인용
"""
    return [base.UserMessage(prompt)]

print("Optional prompt registered: column_review")


## §5. 프롬프트 호출 검증하기

`mcp.list_prompts()`로 등록된 프롬프트 목록을 확인하고, `mcp.get_prompt()`로 인자를 채워 실제 메시지가 어떻게 만들어지는지 살펴봅니다. 이렇게 만들어진 사용자 메시지는 그대로 클로드 API의 메시지 매개변수로 전달이 가능합니다.

In [ ]:
# Week_07.md §2.7 — get_prompt로 프롬프트 메시지 생성 → Claude API에 전달 가능
async def demo_prompt():
    prompts = await mcp.list_prompts()
    print(f"등록된 프롬프트 ({len(prompts)}개):")
    for p in prompts:
        print(f"  - {p.name}: {p.description}")
        for arg in (p.arguments or []):
            req = "필수" if arg.required else "선택"
            print(f"      · {arg.name} ({req}): {arg.description}")

    print("\n=== structural_review 호출 결과 ===")
    result = await mcp.get_prompt(
        "structural_review",
        arguments={
            "member_type": "beam",
            "b": "300", "d": "540",
            "fck": "27", "fy": "400",
        },
    )
    for msg in result.messages:
        print(f"[{msg.role}]")
        print(msg.content.text)

await demo_prompt()


## §6. 도구·리소스·프롬프트 세 가지 컴포넌트의 협업 시연

사용자가 구조 검토용 프롬프트를 선택하면, 클로드는 자동으로 다음 흐름을 수행합니다. 먼저 KDS 요약 리소스를 조회하여 강도감소계수를 확보하고, 다음으로 휨 검토 도구를 호출하면서 필요한 인자는 사용자에게 추가로 질의하며, 마지막으로 KDS 조항 인용 형식으로 한국어 답변을 생성합니다.

```mermaid
sequenceDiagram
    autonumber
    participant 사용자
    participant 클라이언트 as 모델 컨텍스트 프로토콜 클라이언트
    participant API as 클로드 API
    participant 서버 as structural-mcp 서버
    사용자->>클라이언트: 구조 검토 프롬프트를 선택
    클라이언트->>서버: 프롬프트 조회
    서버-->>클라이언트: 검토 절차가 담긴 사용자 메시지 반환
    클라이언트->>API: 메시지와 도구 목록을 함께 전달
    API->>서버: KDS 요약 리소스 조회
    서버-->>API: 강도감소계수 등 요약 반환
    API->>서버: 휨 검토 도구 호출
    서버-->>API: 공칭 휨강도와 강도비와 판정 반환
    API-->>사용자: "만족. KDS 41 17 00 4.3.1에 따라 ..."
```

> [!finding] 세 가지 컴포넌트 통합의 위력
> 프롬프트가 KDS 리소스 조회를 자동 지시하고, 도구 호출 결과를 자동 해석하며, 도메인 형식으로 자동 정리하는 **end-to-end 도메인 자동화**가 단 한 번의 프롬프트 선택으로 작동합니다. 이것이 강의노트 §2.7에서 강조하는 *"한 번 만들면 모든 학생과 프로젝트가 같은 해석 능력을 자연어로 공유"* 의 진짜 의미입니다.


In [ ]:
# Skilljar Track A1 (S6_06_mcp_client.ipynb)이 만든 MCPClient를 import해 통합 데모.
# 이 셀은 Track B1이 통합 structural_mcp.py를 만들었을 때만 실제 실행 가능.
# (Anthropic API key 필요 — .env에 ANTHROPIC_API_KEY 설정)
import sys

skilljar_path = Path("..") / "skilljar"
if skilljar_path.exists():
    sys.path.insert(0, str(skilljar_path.resolve()))

demo_code = '''
# Pseudo-code for the integrated 3-way orchestration.
# Run this cell only after structural_mcp.py is finalized (Stage 5 §7).
import anthropic
from dotenv import load_dotenv
from mcp_client import MCPClient   # Track A1

load_dotenv()
MODEL = "claude-haiku-4-5"

async def run_structural_review(member_type, b, d, fck, fy, As, Mu):
    async with MCPClient(command="uv", args=["run", "structural_mcp.py"]) as client:
        # 1. Prompt → UserMessage
        prompt = await client.get_prompt("structural_review", {
            "member_type": member_type, "b": b, "d": d, "fck": fck, "fy": fy,
        })
        # 2. Tools advertised to Claude
        tools = await client.list_tools()
        claude_tools = [{
            "name": t.name, "description": t.description,
            "input_schema": t.inputSchema,
        } for t in tools]
        # 3. Tool-use loop with Claude
        api = anthropic.Anthropic()
        messages = [{"role": m.role, "content": m.content.text} for m in prompt.messages]
        messages[0]["content"] += f"\\nAs={As} mm^2, Mu={Mu} kN.m"
        while True:
            r = api.messages.create(model=MODEL, max_tokens=2048,
                                    messages=messages, tools=claude_tools)
            if r.stop_reason != "tool_use":
                return r.content[0].text
            for block in r.content:
                if block.type == "tool_use":
                    out = await client.call_tool(block.name, block.input)
                    messages.append({"role": "assistant", "content": r.content})
                    messages.append({"role": "user", "content": [{
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": out[0].text,
                    }]})
'''
print(demo_code)
print("\n[Note] 위 코드는 Track A1의 MCPClient + structural_mcp.py 통합 후 실행 가능.")


## §7. 통합본 저장하기 — 4단계 누적 산출물

단계 5까지의 모든 컴포넌트, 즉 도구 세 개와 리소스 한 개와 프롬프트 두 개를 하나의 파이썬 파일로 내보냅니다. 다음 단계 6에서는 이 파일을 그대로 클로드 코드 명령행 인터페이스(CLI)에 등록하여 실전에서 사용합니다.

In [ ]:
# Stage 5 산출물: 도구 3개 + 리소스 1개 + 프롬프트 2개
py_src = '''"""structural-mcp — Stage 5 (Tools + Resources + Parser + Prompts).

Cumulative build-up of Week_07.md §2.7:
  Stage 1: FastMCP init
  Stage 2: KDS RAG resource
  Stage 3: Strength check tools
  Stage 4: Midas .mgt parser tool
  Stage 5: structural_review + column_review prompts   <-- this file
"""
import json
import math
from pathlib import Path
from typing import Dict, List

from pydantic import Field
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

mcp = FastMCP("StructuralMCP", log_level="ERROR")


@mcp.tool(name="check_flexural_strength", description="RC 보 휨강도 검토 (KDS 41 17 00).")
def check_flexural_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    As: float = Field(description="인장 철근 단면적 (mm^2)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
    Mu: float = Field(description="소요 휨모멘트 (kN.m)"),
) -> str:
    a = As * fy / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn
    return json.dumps({"phi_Mn": round(phi_Mn, 2), "Mu": round(Mu, 2),
                       "DCR": round(Mu / phi_Mn, 3),
                       "check": "OK" if phi_Mn >= Mu else "NG"}, ensure_ascii=False)


@mcp.tool(name="check_shear_strength", description="RC 부재 전단강도 검토 (KDS 41 17 00).")
def check_shear_strength(
    b: float = Field(description="보 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    Av: float = Field(description="전단 보강근 면적 (mm^2)"),
    s: float = Field(description="전단 보강근 간격 (mm)"),
    fy: float = Field(description="전단 보강근 항복강도 (MPa)"),
    Vu: float = Field(description="소요 전단력 (kN)"),
) -> str:
    Vc = (1 / 6) * math.sqrt(fck) * b * d / 1000
    Vs = Av * fy * d / s / 1000
    phi_Vn = 0.75 * (Vc + Vs)
    return json.dumps({"phi_Vn": round(phi_Vn, 2), "Vu": round(Vu, 2),
                       "DCR": round(Vu / phi_Vn, 3),
                       "check": "OK" if phi_Vn >= Vu else "NG"}, ensure_ascii=False)


@mcp.tool(name="parse_midas_mgt", description="Parse a Midas .mgt model file.")
def parse_midas_mgt(file_path: str = Field(description="Absolute path to .mgt")) -> str:
    p = Path(file_path)
    if not p.is_absolute() or not p.exists():
        return json.dumps({"error": "invalid path", "given": str(p)}, ensure_ascii=False)
    sections: Dict[str, List[List[str]]] = {}
    current = None
    for raw in p.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith(";"):
            continue
        if line.startswith("*"):
            current = line[1:].split()[0].upper()
            sections.setdefault(current, [])
        elif current:
            sections[current].append([c.strip() for c in line.split(",")])
    return json.dumps({"node_count": len(sections.get("NODE", [])),
                       "element_count": len(sections.get("ELEMENT", [])),
                       "materials": sections.get("MATERIAL", []),
                       "sections": sections.get("SECTION", [])}, ensure_ascii=False)


@mcp.resource("kds://41-17-00/summary", mime_type="application/json")
def kds_summary() -> str:
    return json.dumps({"title": "KDS 41 17 00",
                       "flexure": {"phi": 0.85, "section": "4.3.1"},
                       "shear": {"phi": 0.75, "section": "4.4"}}, ensure_ascii=False)


@mcp.prompt(name="structural_review",
            description="KDS 41 17 00 기준 RC 부재 설계 검토.")
def structural_review(
    member_type: str = Field(description="부재 종류 (beam/column/slab)"),
    b: float = Field(description="부재 폭 (mm)"),
    d: float = Field(description="유효 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
) -> list[base.Message]:
    prompt = f"""당신은 건축구조 전문가입니다. KDS 41 17 00 기준에 따라 다음 RC {member_type}을 검토하세요.

## 부재 제원
- 종류: {member_type}, b={b} mm, d={d} mm, fck={fck} MPa, fy={fy} MPa

## 절차
1. `kds://41-17-00/summary` 리소스를 먼저 읽어 phi 계수 확인
2. `check_flexural_strength` / `check_shear_strength` 도구로 수치 검토
3. .mgt 모델이 있다면 `parse_midas_mgt`로 미리 분석

## 출력 규약
- 단위: SI (mm/MPa/kN/kN.m)
- 모든 결과에 KDS 조항 번호 인용 (예: \"KDS 41 17 00 4.3.1에 따라 ...\")
- 응력/강도 소수 둘째 자리
- 한국어로 답변, 첫 줄 ✅/❌ 판정
"""
    return [base.UserMessage(prompt)]


@mcp.prompt(name="column_review",
            description="RC 기둥 축력-휨 상호작용 검토 (KDS 41 17 00 4.5).")
def column_review(
    b: float = Field(description="기둥 폭 (mm)"),
    h: float = Field(description="기둥 깊이 (mm)"),
    fck: float = Field(description="콘크리트 압축강도 (MPa)"),
    fy: float = Field(description="철근 항복강도 (MPa)"),
    Pu: float = Field(description="소요 축력 (kN)"),
    Mu: float = Field(description="소요 휨모멘트 (kN.m)"),
) -> list[base.Message]:
    prompt = f"""KDS 41 17 00 4.5 기준으로 RC 기둥 ({b}x{h} mm, fck={fck}, fy={fy})을
Pu={Pu} kN, Mu={Mu} kN.m에 대해 P-M 상호작용으로 검토하라.
조문 인용·SI 단위·첫 줄 ✅/❌ 판정 강제.
"""
    return [base.UserMessage(prompt)]


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

Path("structural_mcp.py").write_text(py_src, encoding="utf-8")
print(f"Saved: {Path('structural_mcp.py').resolve()}")
print(f"Size:  {Path('structural_mcp.py').stat().st_size} bytes")


## §8. 다음 단계 안내 — 단계 6 (클로드 코드에 등록하기)

이제 **`structural-mcp` 서버 본체가 완성**되었습니다. 도구 세 개와 리소스 한 개와 프롬프트 두 개가 모두 한 자리에 모였습니다.

다음 노트북 단계 6에서는 다음 작업을 수행합니다. 첫째, `claude mcp add` 명령 한 줄로 클로드 코드 명령행 인터페이스에 본 서버를 등록합니다. 둘째, 실제 클로드 코드 세션에서 자연어로 *"이 철근콘크리트 보를 검토해줘"* 형태의 요청을 시연합니다. 셋째, 9주차 멀티 에이전트 패턴에서 본 서버가 한 명의 도메인 에이전트로 동작하는 흐름을 안내합니다.

이번 단계의 의의를 다시 정리하면 다음과 같습니다. 첫째, 도메인 워크플로 자체를 프롬프트로 외부화하여 사용자가 긴 명령을 외울 필요가 없게 만들었습니다. 둘째, 도구와 리소스와 프롬프트 세 가지 컴포넌트가 어떻게 협업하는지를 실제 코드로 확인했습니다. 셋째, 한국 건축 실무의 정형화된 도메인 규약을 컴퓨터가 직접 강제하는 패턴을 익혔습니다.

> [!ref] 강의노트 §2.7 단계 ⑥ — 보너스 단계
> 클로드 코드는 모델 컨텍스트 프로토콜의 **소비자(consumer) 측면**을 대표합니다. 본 강의에서는 8주차에서 본격적으로 다루지만, 본 단계에서는 등록 명령과 자연어 시연까지만 미리 체험합니다 (강의노트 라인 1900대 참조).

> [!action] 다음 노트북으로 이동하기 전에 점검할 사항
> 본 노트북에서 저장한 통합본 `structural_mcp.py` 파일에 도구 세 개와 리소스 한 개와 프롬프트 두 개가 모두 정상적으로 들어가 있는지 한 번 더 확인해 주세요. 다음 단계 6에서 클로드 코드에 등록할 때 이 파일이 그대로 사용됩니다.
